In [1]:
from collections import defaultdict
from matplotlib.colors import LogNorm
from pathlib import Path
import jenkspy
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import os
import pandas as pd
import random
import shutil
import string

from country_config import *
from run_config import *
from utils import *

## Check Results of Pre-Calibration Population Validation

In [2]:
run_folder = Path(pre_calibration_run_path)
output_path = run_folder / "output"
analysis_path = Path(pre_calibration_analysis_path)
analysis_path.mkdir(exist_ok=True)

conn = sqlite3.connect(f"{output_path}/pop_validationmonthly_data_0.db")
data = pd.read_sql_query("SELECT * FROM monthly_site_data_district", conn)

month_ids = np.array(sorted(data["monthly_data_id"].unique()))
population_by_month = np.array([
    data[data["monthly_data_id"] == month]["population"].sum()
    for month in month_ids
])

starting_pop = population_by_month[0]
ending_population = np.mean(population_by_month[-12:])  # Average of last 12 months

print(f"Initial population ({initial_year}): 25%: {starting_pop:,.0f} | scaled to 100%: {starting_pop/pre_calibration_population_validation_scale:,.0f}")
print(f"Final population ({calibration_year}): 25%: {ending_population:,.0f} | scaled to 100%: {ending_population/pre_calibration_population_validation_scale:,.0f}")

Initial population (2011): 25%: 5,090,231 | scaled to 100%: 20,360,924
Final population (2024): 25%: 6,619,262 | scaled to 100%: 26,477,049


In [3]:
model_final_population = ending_population / pre_calibration_population_validation_scale
observed_final_population = TARGET_POPULATION_CALIBRATION_YEAR
difference_in_observed_vs_model_final_population = abs(model_final_population - observed_final_population)
percent_error = (model_final_population - observed_final_population) / observed_final_population * 100
print(f"Model final population: {model_final_population:,.0f}")
print(f"Observed final population: {observed_final_population:,.0f}")
print(f"Difference in model and observed population: {difference_in_observed_vs_model_final_population:,.0f}")
print(f"Percent error: {percent_error:.2f}%")

Model final population: 26,477,049
Observed final population: 26,716,457
Difference in model and observed population: 239,408
Percent error: -0.90%
